In [44]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [45]:
df=pd.read_csv("qoute_dataset.csv")

In [46]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [47]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [48]:
df.shape

(3038, 2)

In [49]:
quotes=df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [50]:
quotes=quotes.str.lower()

In [51]:
import string
translator=str.maketrans('','',string.punctuation)
quotes=quotes.apply(lambda x:x.translate(translator))

In [52]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


tokenization ["hello","how","are","you"]

In [53]:
vocab_size=5000 # Reduced vocab_size
tokinizer=Tokenizer(num_words=vocab_size)
tokinizer.fit_on_texts(quotes)

In [54]:
from tensorflow.keras.preprocessing.text import Tokenizer


In [55]:
word_index = tokinizer.word_index
print(len(word_index))
list(word_index.items())[:10]


8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [56]:
sequence = tokinizer.texts_to_sequences(quotes)

In [57]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [58]:
for i in range(3):
  print(sequence[i])


[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [59]:
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [60]:
len(X)

81295

In [61]:
len(y)

81295

In [62]:
max_len = max(len(x) for x in X)
print(max_len)

687


In [63]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [64]:
y = np.array(y)

In [65]:
X_padded.shape

(81295, 687)

In [66]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [67]:
y.shape

(81295,)

In [68]:
y_one_hot.shape

(81295, 5000)

In [69]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense

In [70]:
embedding_dim = 32 # Reduced embedding dimension
rnn_units = 64 # Reduced RNN units

In [71]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [72]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [73]:
rnn_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [74]:
lstm_model=Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len)
  )
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size,activation='softmax'))

In [75]:

lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [76]:
lstm_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [77]:
epochs=10
batch_size=64 # Reduced batch size

In [78]:
history_rnn=rnn_model.fit(
    X_padded,y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 61s 51ms/step - accuracy: 0.0378 - loss: 6.6195 - val_accuracy: 0.0646 - val_loss: 5.9868
Epoch 2/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 56s 49ms/step - accuracy: 0.0752 - loss: 5.8602 - val_accuracy: 0.1001 - val_loss: 5.7380
Epoch 3/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 56s 49ms/step - accuracy: 0.1077 - loss: 5.5061 - val_accuracy: 0.1124 - val_loss: 5.6683
Epoch 4/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 82s 48ms/step - accuracy: 0.1213 - loss: 5.3018 - val_accuracy: 0.1139 - val_loss: 5.6477
Epoch 5/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 55s 48ms/step - accuracy: 0.1319 - loss: 5.1361 - val_accuracy: 0.1188 - val_loss: 5.6428
Epoch 6/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 56s 49ms/step - accuracy: 0.1416 - loss: 4.9797 - val_accuracy: 0.1173 - val_loss: 5.6614
Epoch 7/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 84s 51ms/step - accuracy: 0.1520 - loss: 4.8484 - val_accuracy: 0.1202 - val_loss: 5.6907
Epoch 8/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 56s 49ms/step - accuracy: 0.1593 -

# Task
The `history_lstm` variable was not fully defined due to an interrupted training process. To continue, I'll first re-run the `lstm_model` training to ensure `history_lstm` is complete. Then, I will re-plot the training histories for both the SimpleRNN and LSTM models to visualize their performance. Based on these plots, I will analyze the models for signs of underfitting or overfitting and propose improvements such as adjusting hyperparameters or model architecture.



## Re-Plot Training Histories

### Subtask:
Re-run the cell that plots the training and validation accuracy and loss for both the SimpleRNN and LSTM models. This will allow us to properly visualize the learning curves and evaluate their current performance, now that `history_lstm` should be defined.


**Reasoning**:
The previous LSTM model training was interrupted, leading to `history_lstm` not being defined. I need to re-run the `lstm_model.fit()` method to complete the training and properly populate the `history_lstm` variable, as per the instructions.



In [85]:
history_lstm=lstm_model.fit(
    X_padded,y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 29s 25ms/step - accuracy: 0.0735 - loss: 5.8767 - val_accuracy: 0.0857 - val_loss: 5.8821
Epoch 2/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 29s 26ms/step - accuracy: 0.0906 - loss: 5.7107 - val_accuracy: 0.1006 - val_loss: 5.8117
Epoch 3/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 29s 25ms/step - accuracy: 0.1078 - loss: 5.5528 - val_accuracy: 0.1041 - val_loss: 5.7563
Epoch 4/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 29s 25ms/step - accuracy: 0.1161 - loss: 5.4147 - val_accuracy: 0.1089 - val_loss: 5.7227
Epoch 5/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 30s 27ms/step - accuracy: 0.1226 - loss: 5.2898 - val_accuracy: 0.1150 - val_loss: 5.7074
Epoch 6/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 29s 25ms/step - accuracy: 0.1323 - loss: 5.1707 - val_accuracy: 0.1160 - val_loss: 5.7156
Epoch 7/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 29s 26ms/step - accuracy: 0.1352 - loss: 5.0784 - val_accuracy: 0.1173 - val_loss: 5.7314
Epoch 8/10
1144/1144 ━━━━━━━━━━━━━━━━━━━━ 29s 25ms/step - accuracy: 0.1388 -

In [88]:
lstm_model.save("lstm_model.h5")

In [90]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word


In [91]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [92]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [93]:
seed_text = "what are you"
next_word = predictor(lstm_model,tokinizer,seed_text,max_len)
print(next_word)

are


In [95]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [96]:
seed = "are you a "
generate_text = generate_text(lstm_model,tokinizer,seed,max_len,10)
print(generate_text)


are you a  book is a book is a man who is a


In [97]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokinizer, f)

In [98]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)